In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

In [2]:
from Code import cycle_process, xtb_process, format_change, logfile_process, main, Tool, model_train
import glob, shutil, os 
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from tqdm import tqdm

diene: means 4π

ene: means 2π

In [3]:
PRED_CSV_DIR = r"G:\work\Secondary_Selection\All_possible_DA_ZINC"
PRED_BO_CSV_DIR = r"G:\work\Secondary_Selection\All_possible_DA_ZINC_BO"
PRED_CSV_ONE_DIR = r"G:\work\Secondary_Selection\All_possible_DA_One_ZINC"
SMILES_DIR= r"G:\work\Secondary_Selection\Diene_Ene_Smiles"
CSV_FILE_DIR= r"../Data/Iteration_Data"
QM_MAP = CSV_FILE_DIR + "/" + "all_property_detailSterimol_witharea.pkl" # Copy from smiles_csv

In [13]:
(-528.2660335 + 528.2717615) * 627.5

3.5943199999852027

In [ ]:
NC(=O)c1ncc[nH]c1=O

In [10]:
Chem.MolToSmiles(Chem.MolFromSmiles("O=C(C1=NC=CNC1=O)O"))

'O=C(O)c1ncc[nH]c1=O'

In [8]:
import numpy as np
target_gjfs = glob.glob(r"G:\work\Secondary_Selection\da_improve_1\ts\*.log")
for target_dir in target_gjfs:
    log = logfile_process.Logfile(target_dir)
    atoms, positions = log.symbol_list, log.running_positions[-1]
    title = log.title
    method = log.method
    # atoms, positions = format_change.read_gjf(target_dir)
    positions = -1 * np.array(positions)
    format_change.block_to_gjf(atoms, positions, target_dir, method=method, title=" ".join([str(each) for each in title]))


# 1. DFT Calculate the Reactants

In [4]:
PRED_CSV_DIR = r"G:\work\Secondary_Selection\All_possible_DA_ZINC"
PRED_BO_CSV_DIR = r"G:\work\Secondary_Selection\All_possible_DA_ZINC_BO"
PRED_CSV_ONE_DIR = r"G:\work\Secondary_Selection\All_possible_DA_One_ZINC"
SMILES_DIR= r"G:\work\Secondary_Selection\Diene_Ene_Smiles2"
OLD_SMILES_DIR = r"G:\work\Secondary_Selection\Diene_Ene_Smiles"
CSV_FILE_DIR= r"../Data/Iteration_Data"
QM_MAP = CSV_FILE_DIR + "/" + "all_property_detailSterimol_witharea.pkl" # Copy from smiles_csv

In [19]:
target_dir = SMILES_DIR     # <- The dir you want to save the result
csv_dirs = [r'G:\work\Secondary_Selection\da_improve_1.csv']
# csv_dirs = ["../Data/ZINC_Data/all_selected_smiles2.csv"]                   # <- The dir you saved the reactants

In [40]:
for csv_dir in csv_dirs:
    df = pd.read_csv(csv_dir)
    for row_id, row in df.iterrows():
        row['Diene'] = Chem.MolToSmiles(Chem.MolFromSmiles(row['Diene']))
        row['Ene'] = Chem.MolToSmiles(Chem.MolFromSmiles(row['Ene']))
        df.loc[row_id] = row
    df.to_csv(csv_dir, index=False)

In [41]:
main.collect_smiles_file(csv_dirs = csv_dirs, root_dir=target_dir, sugan=False, old_smiles_csv = OLD_SMILES_DIR + "/" + 'all_smiles.csv')

is_tran_cycloene, CCNC(=O)O[C@@H]1CC/C=C/CCC1
is_tran_cycloene, NC(=O)O[C@@H]1CC/C=C/CCC1


[10:34:39] Molecule does not have explicit Hs. Consider calling AddHs()
[10:34:39] Molecule does not have explicit Hs. Consider calling AddHs()


In [42]:
xtb_process.shift_to_parra(target_dir + "/first_xtb")

In [43]:
main.smiles_DFT_calc(target_dir)

In [44]:
main.error_improve(target_dir, "mol_dft")

100%|██████████| 5/5 [00:00<00:00, 87.72it/s]


In [45]:
main.SPE_DFT_calc(target_dir, save_chk=False)

100%|██████████| 5/5 [00:00<00:00, 78.67it/s]


In [47]:
main.error_improve(target_dir, "mol_dft_eng")

100%|██████████| 5/5 [00:00<00:00, 1250.02it/s]


In [46]:
main.SPE_DFT_calc(target_dir, save_chk=False, opt_name = "mol_dft", eng_name = "mol_dft_eng_solvent", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)")

100%|██████████| 5/5 [00:00<00:00, 78.97it/s]


In [48]:
main.error_improve(target_dir, "mol_dft_eng_solvent")

100%|██████████| 5/5 [00:00<00:00, 1250.84it/s]


In [49]:
main.SPE_DFT_calc(target_dir, save_chk=False, opt_name = "mol_dft", eng_name = "mol_dft_eng_solvent2", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=1,4-dioxane)")

100%|██████████| 5/5 [00:00<00:00, 79.37it/s]


In [7]:
target_dir = 'G:\work\Secondary_Selection\Diene_Ene_Smiles'

In [50]:
main.smiles_result_analysis(target_dir)

5it [00:00, 27.85it/s]


In [51]:
new_smiles_csv = pd.read_csv(SMILES_DIR + "/" + "all_smiles.csv")
old_smiles_csv = pd.read_csv(OLD_SMILES_DIR + "/" + "all_smiles.csv")
final_smiles_csv = pd.concat([old_smiles_csv, new_smiles_csv])
final_smiles_csv.to_csv(OLD_SMILES_DIR + "/" + "all_smiles.csv", index=False)

# 2. Get Descriptors for Reactants

In [ ]:
ALL_PROPERTIES_diene = ["diene_charge_a", "diene_charge_b", "diene_charge_c", "diene_charge_d",
    "diene_max_neighbor_charge", "diene_min_neighbor_charge", "diene_max_charge", "diene_min_charge", "bond_1", "bond2", "bond3", "distance_ad",
    "angle_a_cos", "angle_b_cos", "diene_max_angle_cos", "diene_min_angle_cos", "torsion",
    "diene_homo-1", "diene_homo", "diene_lumo", "diene_lumo+1", "diene_dipole", 
    "max_diene_L", "min_diene_L", "max_diene_B1", "min_diene_B1", "max_diene_B5", "min_diene_B5"]
ALL_PROPERTIES_ene = ["ene_charge_e", "ene_charge_f", "ene_max_neighbor_charge", "ene_min_neighbor_charge", 
    "diene_max_charge", "diene_min_charge", "bond4", 
    "ene_max_angle_cos", "ene_min_angle_cos", "ene_homo-1", "ene_homo", "ene_lumo", "ene_lumo+1", 
    "ene_dipole", "max_ene_L", "min_ene_L", "max_ene_B1", "min_ene_B1", "max_ene_B5", "min_ene_B5"]

In [8]:
smiles_dict=pd.read_csv(SMILES_DIR + "/" + "all_smiles.csv")
smiles_dir=SMILES_DIR
main.smiles_property_generate(file_name='all_property_detailSterimol.csv', smiles_dict=smiles_dict, smiles_dir=smiles_dir)

100%|██████████| 18646/18646 [13:12:38<00:00,  2.55s/it]    


In [9]:
new_des = ['diene_area_0', 'diene_area_1', 'diene_area_2', 'diene_area_3', 'diene_area_4', 'diene_area_5', 'diene_area_6', 'diene_area_7', 
'ene_area_0', 'ene_area_1', 'ene_area_2', 'ene_area_3', 'ene_area_4', 'ene_area_5', 'ene_area_6', 'ene_area_7', ]

In [ ]:
df = pd.read_csv(SMILES_DIR + "/" + "all_property_detailSterimol.csv")
all_properties = [[] for _ in range(len(new_des))]
mol_dir=SMILES_DIR
for idx, row in tqdm(df.iterrows()):
    smiles = row["Smiles"]
    smiles_id = row["Smiles_Id"]
    smiles_type = row['Type']
    title = row['Title']
    title = [int(each) for each in title.split()]
    mol = Chem.MolFromMolFile(mol_dir + '/mol/smilesid_%.5d.mol' % smiles_id, removeHs=False)
    if smiles_type == 'Diene':
        # try:
        for mol, atom_lists, tran_cis in cycle_process.change_position(mol, prop='diene', return_tran_cis=True):
            if atom_lists[0] == title[0] and atom_lists[-1] == title[1]:
                # result = [tran_cis]
                result = model_train.Calc_areas(mol, atom_lists)
                break
        assert len(result) == 8
        [all_properties[each].append(result[each]) for each in range(0,8)]
        [all_properties[each].append(np.nan) for each in range(8,16)]
        # except:
        #     print(smiles_id, title)
    else:
        for mol, atom_lists in cycle_process.change_position(mol, prop='dieno'):
            if atom_lists[0] == title[0] and atom_lists[-1] == title[1]:
                result = model_train.Calc_areas(mol, atom_lists)
                break
            if atom_lists[0] == title[1] and atom_lists[-1] == title[0]:
                result = model_train.Calc_areas(mol, atom_lists)
                result = [result[5], result[4], result[7], result[6], result[1], result[0], result[3], result[2]]
                break
        [all_properties[each].append(np.nan) for each in range(0,8)]
        [all_properties[each].append(result[each - 8]) for each in range(8,16)]
for each_new_des, all_property in zip(new_des, all_properties):
    df[each_new_des] = all_property
df.to_csv(SMILES_DIR + "/" + "all_property_detailSterimol_witharea.csv", index=False)

In [12]:
import pickle
smiles_dir = SMILES_DIR
property_dict=pd.read_csv(SMILES_DIR + "/" + "all_property_detailSterimol_witharea.csv")
# smiles_dict = 
target_map = {}
for idx in range(len(property_dict["Smiles"])):
    smiles_id = property_dict["Smiles_Id"][idx]
    type_name = property_dict["Type"][idx]
    if type_name == "Diene":
        type_id = 1
    elif type_name == "Ene":
        type_id = 0
    title = property_dict["Title"][idx]
    key = "%.5d %d " % (smiles_id, type_id) + title
    if type_id == 1:
        result = [property_dict[property][idx] for property in main.ALL_PROPERTIES_diene + new_des[:8]]
    elif type_id == 0:
        result = [property_dict[property][idx] for property in main.ALL_PROPERTIES_ene + new_des[8:]]

    target_map[key] = result
with open(smiles_dir + "/" + "all_property_detailSterimol_witharea.pkl", "wb") as f:
    pickle.dump(target_map, f)

# 3. Generate the All Cross-Combination Dataset with passed 4π and 2π

In [ ]:
smiles_dir=SMILES_DIR
smiles_csv_dir = SMILES_DIR + "/" + "all_smiles.csv"
target_dir = PRED_CSV_DIR
# target_dir = PRED_BO_CSV_DIR -> Just for Reported BO

In [ ]:
result_diene_csv = pd.read_csv(r"../Data/ZINC_Data/target_diene_passed.csv")
result_ene_csv = pd.read_csv(r"../Data/ZINC_Data/target_ene_passed.csv")
result_diene_csv = result_diene_csv.loc[~np.isnan(result_diene_csv['E/Hatree'].to_numpy())]
result_ene_csv = result_ene_csv.loc[~np.isnan(result_ene_csv['E/Hatree'].to_numpy())]
dienes = set(result_diene_csv["Index"].tolist())
enes = set(result_ene_csv["Index"].tolist())
len(dienes), len(enes)


In [ ]:
# Run It Just When You Generate All Cross-Combination of Reported Bioorthogonal Reactions
dienes = [5944, 18472, 11875, 18500, 12881]
enes = [18429, 18627, 18460, 18619, 18427, 18408, 11950, 18454, 18405,
        11936, 18502, 18611, 18425, 18583, 11955, 18612, 18420, 18457,
        18445, 18620, 18402, 18400, 18451, 11948, 18421, 18100, 18613,
        11932, 11933, 18450, 18558, 18423, 18424, 18447, 18453, 18403,
        18452, 18426]
# The Result are require from 

In [ ]:
smiles_dict = pd.read_csv(smiles_csv_dir)

all_smiles = smiles_dict["Smiles"].to_list()
all_smiles_ids = smiles_dict['Index'].to_list()
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
QM_MAP = smiles_dir + "/" + "all_property_detailSterimol_witharea.pkl"
with open(QM_MAP, "rb") as f:
    target_map = pickle.load(f)
def diene_with_multienes(diene_id, file_name=None):
    result_dict = {"Diene":{}, "Diene_Index":{}, "Ene":{}, "Ene_Index":{}, "Product":{}, "Unreverse":{}, "Title":{}, "Structure_Id":{}}
    result_dict_id = 0
    diene = all_smiles[all_smiles_ids.index(diene_id)]
    all_mols = []
    all_mols_name = []
    all_titles = []
    result_dir = target_dir + "/"
    if not os.path.isdir(result_dir):
        os.mkdir(result_dir)
    for ene_id in enes:
        ene = all_smiles[all_smiles_ids.index(ene_id)]
        diene_conf_id = smiles_dict["Stable_conf_id"][diene_id]
        ene_conf_id = smiles_dict["Stable_conf_id"][ene_id]

        diene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % diene_id
        diene_mol = Chem.MolFromMolFile(diene_mol_dir, removeHs=False)
        diene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (diene_id, diene_conf_id)
        diene_log = logfile_process.Logfile(diene_log_dir)
        atom_list = diene_log.symbol_list
        position = diene_log.first_atom_position
        diene_mol = xtb_process.xtb_to_mol(diene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(diene_mol, 50)
        assert len(diene_mol.GetConformers()) !=0

        ene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % ene_id
        ene_mol = Chem.MolFromMolFile(ene_mol_dir, removeHs=False)
        ene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (ene_id, ene_conf_id)
        ene_log = logfile_process.Logfile(ene_log_dir)
        atom_list = ene_log.symbol_list
        position = ene_log.first_atom_position
        ene_mol = xtb_process.xtb_to_mol(ene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(ene_mol, 50)
        assert len(ene_mol.GetConformers()) !=0
        molgroup = []
        # try:
        molgroup = cycle_process.react(dieno_mol = ene_mol, diene_mol = diene_mol,select_diene=0, FFopt=False)
    
        for structure_id, [eng, mol, title] in enumerate(molgroup):
            diene_atoma, diene_atomb, ene_atomc, ene_atomd, _, title_1 = [int(each) for each in title[:6]]
            diene_key = "%.5d %d %d %d" % (diene_id, 1, diene_atoma, diene_atomb)
            ene_key = "%.5d %d %d %d" % (ene_id, 0, ene_atomc, ene_atomd)
            if diene_key not in target_map.keys() or ene_key not in target_map.keys():
                continue
            Chem.SanitizeMol(mol)
            all_mols.append(mol)
            all_titles.append(title)
            product = Chem.MolToSmiles(mol)
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Diene_Index"][result_dict_id] = diene_id
            result_dict["Ene"][result_dict_id] = ene
            result_dict["Ene_Index"][result_dict_id] = ene_id
            result_dict["Product"][result_dict_id] = product
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Title"][result_dict_id] = " ".join([str(num) for num in title])
            result_dict["Structure_Id"][result_dict_id] = structure_id
            result_dict["Unreverse"][result_dict_id] = title[-1]
            result_dict_id += 1
        # except:
        #     pass
    if file_name == None:
        result_dict_name = result_dir + "Diene_Index_%.5d.csv" % diene_id
    else:
        result_dict_name = result_dir + "%s.csv" % file_name
    result_dict = pd.DataFrame(result_dict)
    result_dict.to_csv(result_dict_name, index=False)





In [ ]:
for diene in tqdm(dienes):
    diene_with_multienes(diene)

In [ ]:
def diene_with_self(diene_indexs, file_name=None):
    result_dict = {"Diene":{}, "Diene_Index":{}, "Ene":{}, "Ene_Index":{}, "Product":{}, "Unreverse":{}, "Title":{}, "Structure_Id":{}}
    result_dict_id = 0
    smiles_csv = pd.read_csv(SMILES_DIR + '/all_smiles.csv')
    smiles_id_dict = {row["Index"]: row["Smiles"] for row_id, row in smiles_csv.iterrows()}
    all_mols = []
    all_mols_name = []
    all_titles = []
    target_dir = r"G:\work\Secondary_Selection\All_possible_DA"
    result_dir = target_dir + "/"
    if not os.path.isdir(result_dir):
        os.mkdir(result_dir)
    for diene_id in diene_indexs:
        # try:
        smiles_dir=SMILES_DIR

        ene_id = diene_id
        diene = smiles_id_dict[diene_id]
        ene = diene
        diene_conf_id = smiles_csv["Stable_conf_id"][diene_id]
        ene_conf_id = smiles_csv["Stable_conf_id"][ene_id]
        if len(cycle_process.find_all_diene(Chem.MolFromSmiles(diene))) == 0:
            continue
        diene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % diene_id
        diene_mol = Chem.MolFromMolFile(diene_mol_dir, removeHs=False)
        diene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (diene_id, diene_conf_id)
        diene_log = logfile_process.Logfile(diene_log_dir)
        atom_list = diene_log.symbol_list
        position = diene_log.first_atom_position
        diene_mol = xtb_process.xtb_to_mol(diene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(diene_mol, 50)
        assert len(diene_mol.GetConformers()) !=0

        ene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % ene_id
        ene_mol = Chem.MolFromMolFile(ene_mol_dir, removeHs=False)
        ene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (ene_id, ene_conf_id)
        ene_log = logfile_process.Logfile(ene_log_dir)
        atom_list = ene_log.symbol_list
        position = ene_log.first_atom_position
        ene_mol = xtb_process.xtb_to_mol(ene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(ene_mol, 50)
        assert len(ene_mol.GetConformers()) !=0
        molgroup = []
        # try:
        molgroup = cycle_process.react(dieno_mol = ene_mol, diene_mol = diene_mol,select_diene=0, FFopt=False)
    
        for structure_id, [eng, mol, title] in enumerate(molgroup):
            Chem.SanitizeMol(mol)
            all_mols.append(mol)
            all_titles.append(title)
            product = Chem.MolToSmiles(mol)
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Diene_Index"][result_dict_id] = diene_id
            result_dict["Ene"][result_dict_id] = ene
            result_dict["Ene_Index"][result_dict_id] = ene_id
            result_dict["Product"][result_dict_id] = product
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Title"][result_dict_id] = " ".join([str(num) for num in title])
            result_dict["Structure_Id"][result_dict_id] = structure_id
            result_dict["Unreverse"][result_dict_id] = title[-1]
            result_dict_id += 1
            # except:
            #     pass
        # except:
        #     continue
    if file_name == None:
        result_dict_name = result_dir + "Diene_Index_%.5d.csv" % diene_id
    else:
        result_dict_name = result_dir + "%s.csv" % file_name
    result_dict = pd.DataFrame(result_dict)
    result_dict.to_csv(result_dict_name)

In [ ]:
diene_with_self(list(dienes | enes), file_name="diene_self")

# 4. DFT Calculation Reactions

In [ ]:
target_dir = r"G:\work\Secondary_Selection\BO_Low_than_24_2"
smiles_dir = r"G:/work/Secondary_Selection/Diene_Ene_Smiles"
smiles_csv_dir = r"G:/work/Secondary_Selection/Diene_Ene_Smiles/all_smiles.csv"
reaction_file_dir = r"G:/work/Secondary_Selection/BO_LowThan_24_2.csv"
Result_csv = "../Data/target_diene_ene_Result.csv"
Times = 1
acquire_title = False

In [52]:
target_dir = r"G:\work\Secondary_Selection\da_improve_1"
smiles_dir = r"G:/work/Secondary_Selection/Diene_Ene_Smiles"
smiles_csv_dir = r"G:/work/Secondary_Selection/Diene_Ene_Smiles/all_smiles.csv"
reaction_file_dir = r"G:/work/Secondary_Selection/da_improve_1.csv"
Result_csv = "../Data/alkyne.csv"
Times = 1
acquire_title = False

In [53]:
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
run_sugan_file = glob.glob("../Data/GaussianSample/run_new.sh")[0]
new_sugan_file = target_dir + "/run_new.sh"
shutil.copy(run_sugan_file, new_sugan_file)

IndexError: list index out of range

In [54]:
main.collect_reaction_file(reaction_file_dir=reaction_file_dir, target_dir=target_dir, smiles_dir=smiles_dir, smiles_csv_dir=smiles_csv_dir, n_cores=60, acquire_title=acquire_title, check_charge=0, max_attemp_num=50, acquire_map=True)

9it [00:21,  2.43s/it]


In [55]:
xtb_process.shift_to_parra(target_dir + "/first_xtb")

In [ ]:
main.smiles_DFT_calc(target_dir, has_title=1)

In [58]:
main.error_improve(target_dir, "mol_dft")

  6%|▋         | 5/80 [00:00<00:03, 20.78it/s]

17514_18654_00000_0000.log 21 22 with a wrong distance 0.4810861439939685
G:\work\Secondary_Selection\da_improve_1/mol_dft\17514_18654_00004_0000.log didn't run successful
17514_18654_00004_0000.log 3 4 with a wrong distance 1.3581667171351746


 12%|█▎        | 10/80 [00:01<00:08,  7.78it/s]

17514_18654_00007_0000.log 3 4 with a wrong distance 1.3395711617560768


 21%|██▏       | 17/80 [00:01<00:06, 10.20it/s]

...

 78%|███████▊  | 62/80 [00:05<00:01, 12.01it/s]

18657_18656_00003_0000.log 26 19 with a wrong distance 0.4837150033375226


 84%|████████▍ | 67/80 [00:05<00:01, 12.11it/s]

18657_18656_00007_0000.log 3 4 with a wrong distance 1.3400521566812975


100%|██████████| 80/80 [00:06<00:00, 13.28it/s]


In [59]:
main.SPE_DFT_calc(target_dir, "mol_dft", "mol_dft_eng")

100%|██████████| 80/80 [00:00<00:00, 134.51it/s]


In [60]:
main.error_improve(target_dir, "mol_dft_eng")

100%|██████████| 80/80 [00:00<00:00, 2580.67it/s]


In [61]:
main.reaction_result_analysis(target_dir)

80it [00:00, 133.75it/s]


In [ ]:
# if Just Calculate deltaG, Run This and Ignore the Code After
target_csv = target_dir + "/Result.csv"
shutil.copy(target_csv, Result_csv)

### Detail Conformer Search

In [62]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.5, Difreeze=1, opt_name="mol_dft", om_name='xtb_om', use_gauss=False)

18658_18100_00000_-0001 Fail in DFT
18658_18100_00001_-0001 Fail in DFT
18658_18100_00002_-0001 Fail in DFT
18658_18100_00003_-0001 Fail in DFT
18658_18100_00008_-0001 Fail in DFT
17514_18656_00000_-0001 Fail in DFT
17514_18656_00001_-0001 Fail in DFT
17514_18656_00002_-0001 Fail in DFT
17514_18656_00003_-0001 Fail in DFT
17514_18656_00004_-0001 Fail in DFT
17514_18656_00007_-0001 Fail in DFT
17514_18654_00000_-0001 Fail in DFT
17514_18654_00001_-0001 Fail in DFT
17514_18654_00002_-0001 Fail in DFT
17514_18654_00003_-0001 Fail in DFT
17514_18654_00004_-0001 Fail in DFT
17514_18654_00007_-0001 Fail in DFT
18657_18100_00000_-0001 Fail in DFT
18657_18100_00001_-0001 Fail in DFT
18657_18100_00002_-0001 Fail in DFT
18657_18100_00003_-0001 Fail in DFT
18657_18100_00004_-0001 Fail in DFT
18657_18656_00000_-0001 Fail in DFT
18657_18656_00001_-0001 Fail in DFT
18657_18656_00002_-0001 Fail in DFT
18657_18656_00003_-0001 Fail in DFT
18657_18656_00007_-0001 Fail in DFT
18657_18654_00000_-0001 Fail

In [63]:
xtb_process.shift_to_parra(target_dir + "/xtb_om")

In [69]:
main.smiles_DFT_calc(target_dir, has_title=1, root_dir='xtb_om', dft_dir='om_3', 
                     method='opt=modredundant freq b3lyp/6-31g(d) em=gd3bj', conf_limit=5, rmsd_limit=1, freeze=1)

mol_str:17514_18654_00000didn't find crest_best!
mol_str:17514_18654_00001didn't find crest_best!
mol_str:17514_18654_00002didn't find crest_best!
mol_str:17514_18654_00003didn't find crest_best!
mol_str:17514_18654_00004didn't find crest_best!
mol_str:17514_18654_00007didn't find crest_best!
mol_str:17514_18656_00000didn't find crest_best!
mol_str:17514_18656_00001didn't find crest_best!
mol_str:17514_18656_00002didn't find crest_best!
mol_str:17514_18656_00003didn't find crest_best!
mol_str:17514_18656_00004didn't find crest_best!
mol_str:17514_18656_00007didn't find crest_best!
mol_str:18655_18100_00000didn't find crest_best!
mol_str:18655_18100_00001didn't find crest_best!
mol_str:18655_18100_00002didn't find crest_best!
mol_str:18655_18100_00003didn't find crest_best!
mol_str:18655_18100_00007didn't find crest_best!
mol_str:18655_18654_00000didn't find crest_best!
mol_str:18655_18654_00001didn't find crest_best!
mol_str:18655_18654_00002didn't find crest_best!
mol_str:18655_18654_

### Normal Process

In [36]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.5, Difreeze=0, opt_name="mol_dft", om_name='om_3', use_gauss=True)

18653_18100_00000_-0001 Fail in DFT
18653_18100_00001_-0001 Fail in DFT
18653_18100_00002_-0001 Fail in DFT
18653_18100_00003_-0001 Fail in DFT
18653_18100_00004_-0001 Fail in DFT
18653_18100_00005_-0001 Fail in DFT
18653_18100_00006_-0001 Fail in DFT
18653_18100_00007_-0001 Fail in DFT
18653_18100_00008_-0001 Fail in DFT
18653_18100_00009_-0001 Fail in DFT
18653_18100_00010_-0001 Fail in DFT
18653_18100_00011_-0001 Fail in DFT
18653_18100_00016_-0001 Fail in DFT
18653_18100_00017_-0001 Fail in DFT
18653_18100_00018_-0001 Fail in DFT
18653_18100_00019_-0001 Fail in DFT


In [ ]:
xtb_process.shift_to_sugan(target_dir + "/xtb_om", 2)

In [15]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.9, Difreeze=1, opt_name="mol_dft", om_name='xtb_om', use_gauss=False)
# xtb_process.shift_to_sugan(target_dir + "/xtb_om", 1)

In [16]:
main.reaction_calc_om_2(target_dir,
                        target_cos=-0.5, 
                        Difreeze=1, 
                        gaussian_input=False, 
                        use_gauss=False, 
                        om_name1='xtb_om',
                        om_name2='xtb_om_2')
# xtb_process.shift_to_sugan(target_dir + "/xtb_om_2", 1)

7it [00:00, 225.42it/s]


In [17]:
main.reaction_calc_om_2(target_dir,
                        target_cos=-0.5, 
                        Difreeze=0, 
                        gaussian_input=False, 
                        use_gauss=True, 
                        om_name1='xtb_om_2',
                        om_name2='om_3',
                        )

7it [00:00, 466.37it/s]


In [52]:
main.error_improve(target_dir, "om_3")

100%|██████████| 38/38 [00:00<00:00, 47.51it/s]


In [ ]:
main.ts_Z_E_check(target_dir, ts_name="om_3")

In [70]:
main.reaction_calc_ts(target_dir, om_name="om_3", )

In [71]:
main.error_improve(target_dir, "ts")

100%|██████████| 80/80 [00:00<00:00, 198.51it/s]


In [72]:
main.reaction_calc_irc(target_dir)

In [ ]:
main.error_improve(target_dir, "irc", 'irc_wrong')

In [167]:
main.reaction_irc_select(target_dir, ts_name='ts')

 12%|█▏        | 51/443 [00:00<00:06, 63.66it/s]

G:\work\Secondary_Selection\BO_Low_than_24_2/ts\05944_18025_00003_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\05944_18025_00004_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\05944_18025_00006_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\05944_18025_00009_0000.log did't find right irc files


 56%|█████▌    | 248/443 [00:05<00:04, 39.30it/s]

G:\work\Secondary_Selection\BO_Low_than_24_2/ts\14246_18025_00000_0000.log did't find right irc files


 83%|████████▎ | 368/443 [00:07<00:01, 47.15it/s]

G:\work\Secondary_Selection\BO_Low_than_24_2/ts\17291_18025_00000_0000.log did't find right irc files


 96%|█████████▌| 426/443 [00:08<00:00, 56.43it/s]

G:\work\Secondary_Selection\BO_Low_than_24_2/ts\18097_17550_00000_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\18097_17550_00001_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\18097_17550_00002_0000.log did't find right irc files
G:\work\Secondary_Selection\BO_Low_than_24_2/ts\18097_17550_00003_0000.log did't find right irc files


100%|██████████| 443/443 [00:09<00:00, 47.71it/s]


In [ ]:
main.ts_Z_E_check(target_dir, ts_name="ts")

In [73]:
main.ts_SPE_DFT_calc(target_dir, 'ts')

100%|██████████| 39/39 [00:00<00:00, 95.62it/s]


In [78]:
main.error_improve(target_dir, "ts_eng")

100%|██████████| 80/80 [00:00<00:00, 1879.85it/s]


In [74]:
main.ts_SPE_DFT_calc(target_dir, eng_name = "ts_eng_solvent", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)")

100%|██████████| 39/39 [00:00<00:00, 101.78it/s]


In [77]:
main.error_improve(target_dir, "ts_eng_solvent")

100%|██████████| 80/80 [00:00<00:00, 2062.53it/s]


In [ ]:
main.check_product_stereo(target_dir)

In [75]:
main.reaction_calc_seperate_reactants(target_dir, type='both')

In [76]:
main.error_improve(target_dir, 'sep_react')

100%|██████████| 80/80 [00:00<00:00, 1455.63it/s]


In [ ]:
target_dir = r"G:\work\Secondary_Selection\pre_diene_reaction"

In [ ]:
main.ts_Z_E_check(target_dir, ts_name='ts', move_file='wrong_ZEH', ignore_H=False)

In [ ]:
main.check_product_stereo(target_dir)

In [ ]:
main.ts_up_down_check(target_dir, ts_name='ts', dust_bin="up_down_fail_ts")

In [ ]:
main.rotated_reactant(target_dir)

In [ ]:
main.error_improve(target_dir, 'special_reactant')

In [ ]:
main.ts_SPE_DFT_calc(target_dir, "special_reactant", "special_reactant_eng")

In [ ]:
main.error_improve(target_dir, 'special_reactant_eng')

In [79]:
target_dict = main.reaction_ts_result_analysis(target_dir, write_title=0, ts_eng_name='ts_eng', om_name='om_3')
target_dict

80it [00:00, 90.51it/s] 


In [ ]:
target_csv = target_dir + "/Result_.csv"
shutil.copy(target_csv, Result_csv)

In [45]:
main.ts_SPE_DFT_calc(target_dir, ts_name='Test2/ts', eng_name = "Test2/ts_eng_solvent", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)")

100%|██████████| 4/4 [00:00<00:00, 70.05it/s]


In [ ]:
# target_dict = main.reaction_distort_interaction_analysis(target_dir)
# target_dict.to_csv(target_dir + "/Result.csv", index=0)

# 3. Generate the All Cross-Combination Dataset with passed 4π and 2π

In [ ]:
smiles_dict = pd.read_csv(r"G:\work\Secondary_Selection\Diene_Ene_Smiles\all_smiles.csv")
smiles_dir=r"G:\work\Secondary_Selection\Diene_Ene_Smiles"
all_smiles = smiles_dict["Smiles"].to_list()
all_smiles_ids = smiles_dict['Index'].to_list()
root_dir = r"G:\work\Secondary_Selection/All_possible_DA_ZINC_BO"
result_dict = {"Diene":{}, "Diene_Index":{}, "Ene":{}, "Ene_Index":{}, "Product":{}, "Unreverse":{}, "Title":{}, "Structure_Id":{}}
result_dict_id = 0
file_name = "Real_BO"
for row_id, row in pd.read_csv(r"G:\work\Secondary_Selection\DA0_clear.csv").iterrows():
    diene = row["Diene"]
    ene = row['Ene']
    diene_id= row["Diene_Index"]
    ene_id = row["Ene_Index"]
    diene_conf_id = smiles_dict["Stable_conf_id"][diene_id]
    ene_conf_id = smiles_dict["Stable_conf_id"][ene_id]

    diene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % diene_id
    diene_mol = Chem.MolFromMolFile(diene_mol_dir, removeHs=False)
    diene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (diene_id, diene_conf_id)
    diene_log = logfile_process.Logfile(diene_log_dir)
    atom_list = diene_log.symbol_list
    position = diene_log.first_atom_position
    diene_mol = xtb_process.xtb_to_mol(diene_mol, [atom_list], [position], 1)
    # Chem.AllChem.EmbedMultipleConfs(diene_mol, 50)
    assert len(diene_mol.GetConformers()) !=0

    ene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % ene_id
    ene_mol = Chem.MolFromMolFile(ene_mol_dir, removeHs=False)
    ene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (ene_id, ene_conf_id)
    ene_log = logfile_process.Logfile(ene_log_dir)
    atom_list = ene_log.symbol_list
    position = ene_log.first_atom_position
    ene_mol = xtb_process.xtb_to_mol(ene_mol, [atom_list], [position], 1)
    # Chem.AllChem.EmbedMultipleConfs(ene_mol, 50)
    assert len(ene_mol.GetConformers()) !=0
    molgroup = []
    # try:
    molgroup = cycle_process.react(dieno_mol = ene_mol, diene_mol = diene_mol,select_diene=200, FFopt=False)

    for structure_id, [eng, mol, title] in enumerate(molgroup):
        diene_atoma, diene_atomb, ene_atomc, ene_atomd, _, title_1 = [int(each) for each in title[:6]]
        diene_key = "%.5d %d %d %d" % (diene_id, 1, diene_atoma, diene_atomb)
        ene_key = "%.5d %d %d %d" % (ene_id, 0, ene_atomc, ene_atomd)
        if diene_key not in target_map.keys() or ene_key not in target_map.keys():
            continue
        Chem.SanitizeMol(mol)
        product = Chem.MolToSmiles(mol)
        result_dict["Diene"][result_dict_id] = diene
        result_dict["Diene_Index"][result_dict_id] = diene_id
        result_dict["Ene"][result_dict_id] = ene
        result_dict["Ene_Index"][result_dict_id] = ene_id
        result_dict["Product"][result_dict_id] = product
        result_dict["Diene"][result_dict_id] = diene
        result_dict["Title"][result_dict_id] = " ".join([str(num) for num in title])
        result_dict["Structure_Id"][result_dict_id] = structure_id
        result_dict["Unreverse"][result_dict_id] = title[-1]
        result_dict_id += 1
    # except:
    #     pass
result_dict_name = root_dir + "\%s.csv" % file_name
result_dict = pd.DataFrame(result_dict)
result_dict.to_csv(result_dict_name, index=False)

# 计算能量与实验能量比较

In [ ]:
csv1 = pd.read_csv(r"D:\work\Data\DI_process\csv_files\DFT_EXP.csv")
csv2 = pd.read_csv(r"D:\work\Data\DI_process\csv_files\DI_result_random_7_bond_ab.csv")

In [ ]:
all_smiles_ids = set(csv1["Diene_Index"].to_list() + csv2["Diene_Index"].to_list() + csv1["Ene_Index"].to_list() + csv2["Ene_Index"].to_list())
all_smiles_ids = set(csv2["Diene_Index"].to_list() + csv2["Ene_Index"].to_list())

In [ ]:
import model_train
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
csv = pd.read_csv(r"D:\work\Data\DI_process\csv_files\DFT_EXP.csv")
calc_value = csv["deltaGa(Solvent)"][2:].to_numpy()
exp_value = csv["DFT_EXP"][2:].to_numpy()
min_ = 0
max_ = 40
x = exp_value
y = calc_value

r2 = model_train.r2_score(x, y)

# 绘制散点图
plt.figure(figsize=(4, 4))
plt.xlim(min_, max_)
plt.ylim(min_, max_)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
# plt.xlabel("Real", fontsize=18)
# plt.ylabel("Prediction", fontsize=18)
# if title != None:
#     plt.title("%s\nR2:%.3f, MAE:%.3f, MSE:%.3f" % (title, r2, mae, mse), fontsize=24)
#     plt.title("%s"%title,fontsize=24)
z = np.linspace(min_, max_, 10000)
plt.plot(z, z, c='gray')


# 添加回归性能指标到图像的第二行
model = LinearRegression()
model.fit(exp_value.reshape(-1, 1), calc_value)
plt.plot([min_, max_], model.predict([[min_], [max_]]), c='blue')
plt.scatter(x, y, marker="*", c="g", s=200)

# 显示图像
plt.savefig("test.svg", format="svg")
plt.show()

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt

In [ ]:
model.predict(exp_value.reshape(-1, 1))

In [ ]:
model.score(exp_value.reshape(-1, 1), calc_value)

In [ ]:
model.coef_, model.intercept_, model.rank_

In [ ]:
model = LinearRegression()
model.fit(exp_value.reshape(-1, 1), calc_value)
model.score(exp_value.reshape(-1, 1), calc_value)